# Etapa 1 — Scraping y extracción de artículos

Este notebook orquesta el pipeline de recolección de datos.
Toda la lógica está en `src/` para que este archivo sea limpio y fácil de leer.

**Pipeline:**
1. Buscar noticias en Google News
2. Resolver URLs reales (resolver redirects de Google)
3. Filtrar por medios objetivo y elegir la más relevante por medio
4. Extraer el texto completo de cada artículo
5. Guardar en disco para no tener que re-scrapear

## Setup
Asegurate de haber instalado las dependencias primero:
```bash
pip install -r requirements.txt
```

In [ ]:
import sys
sys.path.insert(0, '../src')  # Para que Python encuentre los módulos en src/

from scraper import buscar_noticias
from extractor import extraer_texto_articulos
from storage import guardar_articulos, cargar_articulos, listar_busquedas_guardadas

## 1. Buscar noticias
Ingresá el tema que querés analizar. El scraper busca en Google News (en español, región AR),
recorre varias páginas y resuelve los redirects para identificar correctamente el dominio de cada artículo.

In [ ]:
query = input("Tema a buscar: ")

# buscar_noticias devuelve la noticia más relevante de cada medio objetivo
noticias = buscar_noticias(query, verbose=True)

## 2. Extraer texto completo
Para cada artículo encontrado, descargamos la página y extraemos el cuerpo del texto
usando `trafilatura`. Los artículos con paywall o estructura incompatible se descartan.

In [ ]:
articulos = extraer_texto_articulos(noticias, verbose=True)

## 3. Inspección rápida del contenido extraído

In [ ]:
for art in articulos:
    print(f"{'='*60}")
    print(f"MEDIO:    {art['dominio']}")
    print(f"TÍTULO:   {art['title']}")
    print(f"URL:      {art['link_real']}")
    print(f"PALABRAS: {len(art['texto'].split())}")
    print(f"\nPRIMEROS 300 CARACTERES:")
    print(art['texto'][:300])
    print()

## 4. Guardar en disco
Guardamos los artículos en `data/raw/` como JSON. El nombre del archivo incluye
la fecha y el tema buscado para tener un historial organizado.

In [ ]:
ruta_guardado = guardar_articulos(articulos, query)

## 5. (Opcional) Cargar una búsqueda anterior
Si querés trabajar con datos ya scrapeados sin volver a correr el pipeline:

In [ ]:
# Ver todas las búsquedas guardadas
busquedas = listar_busquedas_guardadas()
for b in busquedas:
    print(b)

In [ ]:
# Cargar la más reciente (o especificar la ruta que quieras)
if busquedas:
    datos = cargar_articulos(busquedas[0])
    print(f"Query original: {datos['query']}")
    print(f"Fecha: {datos['fecha_busqueda']}")
    print(f"Artículos: {datos['cantidad_articulos']}")
    articulos_cargados = datos['articulos']